# 第04讲中文代码说明

原版另存；原输出为历史记录，不冒充本次运行。
注释仅解释原行为，保留Markdown源码节选缺陷；原7B量化加载未运行。

## 单元 0：为什么读源码

【建议会】这是Markdown学习动机。引用源码能澄清自动注入、默认目标和冻结；文字“后续直接改源码”是参考资料描述，不是本次要求修改第三方库。此次没有修改官方源码，只另生成教学注释文件。

- 源码面前，了无秘密
    - 学习成熟的框架，提升代码水平
    - 后续直接改源码

## 单元 1：type 标题

【必须会】Markdown标题引出三种类型。它不调用type()，也不创建Python类型变量。

## type

## 单元 2：任务映射与适配方法映射

【必须会】两个围栏均为字典源码节选，Markdown不会执行。键选择实现类；值为**类对象**，不是已创建的模型实例。冒号是键值对应，逗号分隔项，大括号构造dict；下划线/大写名字表示常量命名习惯，不保证不可修改。

第一张映射：SEQ_CLS对应序列分类包装；SEQ_2_SEQ_LM对应编码器-解码器语言模型包装；CAUSAL_LM对应因果LM包装；TOKEN_CLS对应逐token分类包装。虽然常量叫MODEL_TYPE，它在这里按task_type选择任务外层，不是按bloom/llama架构选择。

第二张映射：PeftType.LORA→LoraModel、PROMPT_TUNING→PromptEmbedding、P_TUNING→PromptEncoder、PREFIX_TUNING→PrefixEncoder、ADALORA→AdaLoraModel、ADAPTION_PROMPT→AdaptionPromptModel。PeftType为Enum，`.LORA`为枚举成员；各方法的实现和参数形态不同，不能把prompt embedding也称为LoRA矩阵。

两段依赖已导入各类名和PeftType，原节选未给这些import。在隔离字典上下文补入固定官方类，真实执行两段并与v0.3.0映射逐项比较；不是实例化全部适配方法。真正get_peft_model先按任务创建包装，再在非prompt学习路径选择LoraModel，详见主文。


- **task_type/model_type**:

    
    - `LoraConfig(task_type='CAUSAL_LM')`
    
    ```
    MODEL_TYPE_TO_PEFT_MODEL_MAPPING = {
        "SEQ_CLS": PeftModelForSequenceClassification,
        "SEQ_2_SEQ_LM": PeftModelForSeq2SeqLM,
        "CAUSAL_LM": PeftModelForCausalLM,
        "TOKEN_CLS": PeftModelForTokenClassification,
    }
    ```

- **peft_type**

    ```
    PEFT_TYPE_TO_MODEL_MAPPING = {
        PeftType.LORA: LoraModel,
        PeftType.PROMPT_TUNING: PromptEmbedding,
        PeftType.P_TUNING: PromptEncoder,
        PeftType.PREFIX_TUNING: PrefixEncoder,
        PeftType.ADALORA: AdaLoraModel,
        PeftType.ADAPTION_PROMPT: AdaptionPromptModel,
    }
    ```

## 单元 3：PeftModel 标题

【建议会】Markdown标题中的反引号是展示代码样式；本格没有调用PeftModel构造器。

## `PeftModel`

## 单元 4：包装对象的初始化属性

【必须会】这段依赖方法内部self、传入的model和adapter_name。self.base_model=model保存对象引用；self.config=self.base_model.config复用配置引用；active_adapter保存当前adapter名字，通常为'default'。字符串'default'是adapter标识，不是任务CAUSAL_LM。

这三条赋值不复制基权重，也没有完成LoRA注入。v0.3.0完整PeftModel.__init__随后把base_model进一步包装为LoraModel。只看这个节选不能以为最后base_model永远是裸BLOOM。

普通顶层会因self未定义失败，故原围栏不是可独立运行文件。新增验证在明确隔离的属性上下文检查引用关系，同时实际调用官方构造器检查最终外层→LoraModel→原BLOOM结构，二者的证据范围分开记录。没有Tensor运算或shape变换。

```
self.base_model = model
self.config = self.base_model.config
# adapter_name="default"
self.active_adapter = adapter_name
```

## 单元 5：配置与注入模块标题

【必须会】Markdown对比LoraConfig与LoraModel：前者存超参数，后者管理注入和adapter；二者不是同一个对象，也不是两个顺序运行的大模型。

## LoraConfig 与 LoraModel

## 单元 6：LoraConfig 标题

【建议会】引出数据类源码节选。标题不定义类，不运行dataclass装饰器。

### LoraConfig

## 单元 7：不完整的 dataclass 源码

【必须会】@dataclass为装饰器，给类生成__init__等方法；LoraConfig继承PeftConfig；冒号为类型注解，field保存默认值/metadata。metadata中的help是说明，不是算法或校验代码。类型注解不会自动验证传入参数。

| 源字段 | 原节选含义 | 限制 |
|---|---|---|
| r:int=field(default=8,...) | 默认瓶颈8，int配置 | 不是后面真正配置的r=16 |
| target_modules:Optional[Union[List[str],str]] | 列表/正则字符串/None的类型说明 | 原节选没有默认值赋值；Optional不自动等于default=None |
| lora_alpha:int=field(default=None,...) | 原节选默认None | 使用前应给有效数值或看完整版本处理，不能把None当1 |
| lora_dropout:float=field(default=None,...) | 原节选默认None | 不等于概率0 |

独立运行还缺dataclass、field、typing及PeftConfig导入。即使补这些真实依赖，本节选也在类装饰阶段报`TypeError: non-default argument 'target_modules' follows default argument`：有默认的r后面又出现未设默认的target_modules。

固定官方v0.3.0完整定义实际给target_modules加了field(default=None,...)，还有fan_in_fan_out、bias、modules_to_save、init_lora_weights和__post_init__等。后者设置peft_type=LORA。节选缺少这些内容，不能把本格当成完整类或注入结果。中文注释版只标明缺陷，保留原字句/AST，不悄悄补默认值。

```
@dataclass
class LoraConfig(PeftConfig):
    r: int = field(default=8, metadata={"help": "Lora attention dimension"})
    target_modules: Optional[Union[List[str], str]]
    lora_alpha: int = field(default=None, metadata={"help": "Lora alpha"})
    lora_dropout: float = field(default=None, metadata={"help": "Lora dropout"})
    
```

## 单元 8：默认目标字典与模块扫描

【必须会】本格两段源码分别是架构映射与扫描逻辑。开头“default:query_key_value”只适用于BLOOM等对应架构，不是全部模型的统一默认。

目标字典的值为list[str]：t5/mt5用q/v；bart、opt、gptj、gpt_neo、llama用q_proj/v_proj；gpt2用c_attn；bloom、gpt_neox、chatglm用query_key_value；blip-2同时列q/v/q_proj/v_proj；bert、roberta、xlm-roberta、electra、layoutlm用query/value；deberta-v2用query_proj/value_proj；deberta用in_proj。中文注释逐项解释其为各架构的模块名，不能在BLOOM上换成LLaMA名字。

扫描中的self是LoraModel，self.model是原模型。named_modules迭代路径与模块对象，列表推导式保存key路径快照；`_`忽略模块值。先快照后替换避免遍历模块树时直接改变遍历集合。它不同于named_parameters的参数名字与Parameter。

target_modules为str时，re.fullmatch返回Match或None，匹配**完整路径**；为列表时，生成器逐项检查key.endswith(target_key)，any将布尔结果归并。因此裸正则'query_key_value'不匹配带transformer前缀的完整路径，而列表['query_key_value']能匹配后缀；正则'.*query_key_value'可匹配。

原扫描节选依赖self/lora_config/re，而且只得到target_module_found，不包含后续真正替换。匹配失败应结合完整库的ValueError，不是这段else自己抛错。本次用真实BLOOM模块树执行原扫描片段，比较列表、完整正则、裸正则及错误目标的结果；未知/找不到目标的真实get_peft_model路径另行验证。

以H=4096融合QKV为例，原weight为[12288,4096]，r=16时A为[16,4096]、B为[12288,16]，输入[B,S,4096]经过小支路变为[B,S,12288]。本格字典/扫描本身不做这些矩阵乘法；真正前向在库Linear或Linear8bitLt内。

- `target_modules`
    - The names of the modules to apply Lora to.
    - default: `query_key_value`
        - `'transformer.h.0.self_attention.query_key_value'`
    - 都意味着一个大矩阵 $W\in \mathbb R^{A\times B}$；
    


```
TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING = {
    "t5": ["q", "v"],
    "mt5": ["q", "v"],
    "bart": ["q_proj", "v_proj"],
    "gpt2": ["c_attn"],
    "bloom": ["query_key_value"],
    "blip-2": ["q", "v", "q_proj", "v_proj"],
    "opt": ["q_proj", "v_proj"],
    "gptj": ["q_proj", "v_proj"],
    "gpt_neox": ["query_key_value"],
    "gpt_neo": ["q_proj", "v_proj"],
    "bert": ["query", "value"],
    "roberta": ["query", "value"],
    "xlm-roberta": ["query", "value"],
    "electra": ["query", "value"],
    "deberta-v2": ["query_proj", "value_proj"],
    "deberta": ["in_proj"],
    "layoutlm": ["query", "value"],
    "llama": ["q_proj", "v_proj"],
    "chatglm": ["query_key_value"],
}
```

```
key_list = [key for key, _ in self.model.named_modules()]
for key in key_list:
    if isinstance(lora_config.target_modules, str):
        target_module_found = re.fullmatch(lora_config.target_modules, key)
    else:
        target_module_found = any(key.endswith(target_key) for target_key in lora_config.target_modules)
```



## 单元 9：真正构造配置对象

【必须会】from peft导入已安装/已选源码路径中的完整LoraConfig，而不是单元7未执行的Markdown类。实例config字段为r16、alpha32、dropout.05、bias none、task CAUSAL_LM；scale=2。没有真正模型输入/输出Tensor。

target_modules那行仍是注释，q_proj/v_proj不生效，默认由架构bloom映射融合QKV。本格默认构造可训练adapter配置，未进入forward、冻结或替换。metadata帮助手段不影响此处数值。

In [5]:
from peft import LoraConfig 
config = LoraConfig(
    r=16, #low rank
    lora_alpha=32, #alpha scaling， scale lora weights/outputs
    # target_modules=["q_proj", "v_proj"], #if you know the 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # set this for CLM or Seq2Seq
)

## 单元 10：model 到 loramodel 标题

【建议会】Markdown标题表示接下来加载及包装流程。箭头不是Python运算，不复制权重。

### model => loramodel

## 单元 11：导入 Auto 接口

【必须会】导入AutoTokenizer、AutoConfig、AutoModelForCausalLM三个类；实际后续只用最后一个加载。前两者在本notebook未被调用，不能据导入宣称词表/config另下载成功。导入不构造7B模型。

In [2]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

## 单元 12：7B 量化加载

【必须会】AutoModelForCausalLM.from_pretrained为类方法，模型ID为bigscience/bloom-7b1，load_in_8bit=True请求旧式bitsandbytes量化，device_map='auto'自动设备分派。返回BLOOM因果LM对象，并可能联网/缓存/分配设备内存；原代码没有model revision。

8bit不表示所有参数或激活均int8；auto不等于数据并行world_size。加载的参数与结构让后面库决定用Linear8bitLt适配路径。历史下载widget不是这次执行证据。本次未执行原单元12，改用独立补充代码构造小型CPU随机BLOOM，非量化、无远程下载。

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "bigscience/bloom-7b1", 
    load_in_8bit=True, 
    device_map='auto',
)

Overriding torch_dtype=None with `torch_dtype=torch.float16` due to requirements of `bitsandbytes` to enable model loading in mixed int8. Either pass torch_dtype=torch.float16 or don't pass this argument at all to remove this warning.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 单元 13：查看原模型

【建议会】末尾model让notebook显示repr；普通脚本不自动显示。此时它是裸BLOOM对象引用，没有forward、注入或clone。实际CPU验证执行该表达式但在明确的教学模型上下文，不能称原7B加载成功。

In [6]:
model

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(250880, 4096)
    (word_embeddings_layernorm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-29): 30 x BloomBlock(
        (input_layernorm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear8bitLt(in_features=4096, out_features=12288, bias=True)
          (dense): Linear8bitLt(in_features=4096, out_features=4096, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear8bitLt(in_features=4096, out_features=16384, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear8bitLt(in_features=16384, out_features=4096, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((4096,), eps=1e-05, eleme

## 单元 14：导入注入函数

【必须会】get_peft_model是包级普通函数，非model实例方法；输入为模型模块与PeftConfig，返回PEFT包装。导入本身不改变模型，也不定义另一套随机权重。

In [4]:
from peft import get_peft_model

## 单元 15：真正执行 LoRA 注入

【必须会】调用get_peft_model(model,config)，赋回model名字。原BLOOM内的目标子层被改造，外层为PeftModelForCausalLM，中间base_model为LoraModel，其model指向原BLOOM；包装不是三个完整网络，也不是深拷贝。

官方v0.3.0的_find_and_replace找到目标，_replace_module用setattr改变父属性，并把旧weight/bias对象引用交给新层。mark_only_lora_as_trainable将非lora_参数冻结。bias='none'不训练原偏置，但前向仍使用它。

本次直接运行固定v0.3.0官方路径，记录真实函数调用、目标子模块类型变化、旧weight/bias的对象身份保持、原BLOOM身份保持及可训练参数集合。CPU普通Linear路径通过；不能据此宣称Linear8bitLt/CUDA kernel验证成功。

In [7]:
model = get_peft_model(model, config)

## 单元 16：查看注入后模型

【必须会】表达式model显示包装和lora_A/lora_B等模块。repr有助于查看层次，但不能证明参数值已训练、已保存adapter或权重requires_grad全部正确。本格没有loss/backward/optimizer。

固定源码CPU验证将两层H16融合QKV替换成真实适配Linear，A[16,16]、B[48,16]，scale2；新增2048个可训练参数。这是小模型尺寸，与原7B的4096/12288/30层不同，原尺寸对应的参数推导见主文。

In [8]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(250880, 4096)
        (word_embeddings_layernorm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-29): 30 x BloomBlock(
            (input_layernorm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): Linear8bitLt(
                in_features=4096, out_features=12288, bias=True
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=12288, bias=False)
                )
                (lora_embedding_A): Para

## 单元 17：空代码格

【可以暂时不掌握】原source为空，outputs也为空。保留空格及原元数据，说明版不删它；中文注释版仅加入“原为空，没有运算”的注释，AST仍是空Module。不能把原空格算作模型运行或补写训练代码。